> **QAC reproduction record** — claim `femoco-low-2025-li76`, reproduction `qdk-sossa-2026-11`.
>
> This notebook is the executable evidence behind the QAC assessment of Low et al. (2025), *Phys. Rev. X* **15**, 041016.
> It is kept readable on purpose: the markdown walks through the method so it can be used for learning.
> The **last cell writes `results.json`**, the only source of the numbers the public record attributes to QAC;
> nothing in that file is typed by hand. The exact environment is pinned in `requirements.lock.txt`,
> and `run.json` records who executed it and when. Re-execute with `qacclaims execute`.


# Sum-of-squares spectrum amplification for chemistry

This notebook demonstrates how to simulate and estimate the resources for quantum phase estimation (QPE) using a **sum-of-squares spectrum amplification (SOSSA)** block encoding.

The method follows [Low et al. (2025)](https://journals.aps.org/prx/abstract/10.1103/pb2g-j9cw). For the chemical systems studied in that paper, combining spectrum amplification with an optimized tensor factorization reduced the estimated cost by orders of magnitude compared with earlier methods.

QPE estimates an eigenvalue—here, an energy—of a Hamiltonian encoded in a unitary operator. If you are new to QPE, block encoding, or linear combinations of unitaries (LCU), start with the [stretched $\mathrm{N}_2$ notebook](qpe_stretched_n2.ipynb). This notebook focuses on the additional ideas needed for SOSSA and on the paper's **double-factorized tensor hypercontraction (DFTHC)** representation.

You will:

1. represent a chemistry Hamiltonian as a sum of squares using DFTHC, double factorization (DF), or synthetic tensors;
2. build SOSSA phase-estimation circuits with different state-preparation and table-lookup primitives;
3. validate the circuit with a $\mathrm{H}_2$ example and compare the simulated energy with the reference result; and
4. estimate the logical and physical resources for computing the energy of the 76-orbital $\mathrm{FeMoco}$ active space on different quantum hardware backends.

Install the dependencies with:

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```


In [1]:
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

## Why spectrum amplification?

### Qubitization

A Hamiltonian is not unitary, so a **block encoding** places a rescaled copy of it inside a larger unitary $U$ acting on the system plus ancillas:

$$
\left(\langle0|_{a}\otimes\mathbb{I}\right)U\left(|0\rangle_{a}\otimes\mathbb{I}\right)=\frac{H}{\lambda},\qquad \lambda\ge\lVert H\rVert.
$$

The usual construction is a linear combination of unitaries (LCU): PREPARE loads the weights of $H=\sum_j\alpha_jP_j$ into the ancillas, SELECT applies each Pauli string $P_j$, and PREPARE is undone. Then $\lambda=\sum_j|\alpha_j|$, which is large for molecules.

Adding a reflection about $|0\rangle_a$ turns $U$ into a walk operator $W$ whose eigenphases encode the energies:

$$
\cos\theta=\frac{E}{\lambda}.
$$

Phase estimation measures $\theta$. Resolving $E$ to a standard deviation $\sigma_{\mathrm{PEA}}$ takes

$$
p=\left\lceil\frac{\pi\lambda}{2\sigma_{\mathrm{PEA}}}\right\rceil
$$

applications of $W$. The cost grows linearly with $\lambda$, because every energy is squeezed into $[-\lambda,\lambda]$. The [stretched $\mathrm{N}_2$ notebook](qpe_stretched_n2.ipynb) uses this construction in its LCU example.

### Sum-of-squares spectrum amplification (SOSSA)

SOSSA changes the object being encoded. Suppose we can write the Hamiltonian, up to a constant,
as a sum of squares,

$$
H-E_{\mathrm{SOS}}=H_{\mathrm{sqrt}}^{\dagger}H_{\mathrm{sqrt}},
$$

with $H-E_{\mathrm{SOS}}$ positive semi-definite. Block-encoding the rectangular operator
$H_{\mathrm{sqrt}}$ and reflecting about the inner ancillas produces a walk whose eigenphase
$\theta$ tracks $\sqrt{E-E_{\mathrm{SOS}}}$ rather than $E$. Near the bottom of the spectrum the
square root is steep, $\sqrt{x}\gg x$ when $x\ll1$, so states that are close in energy end up far
apart in phase. For the state being measured, the raw normalization $\Lambda$ of the block encoding
is replaced by a smaller **effective normalization**

$$
\lambda_{\mathrm{eff}}=\sqrt{E_{\mathrm{gap}}\left(2\Lambda-E_{\mathrm{gap}}\right)}\ \le\ \Lambda,
\qquad
E_{\mathrm{gap}}=E_{0}-E_{\mathrm{SOS}},
$$

where $E_0$ is the ground-state energy. $\lambda_{\mathrm{eff}}$ takes the place of $\lambda$ in the
query count formula above. Computing $\lambda_{\mathrm{eff}}$ requires an approximate $E_0$ in advance. A Hartree-Fock or CASCI
estimate is sufficient because it only determines the circuit size and does not bias the energy returned by QPE.

## Getting to a sum of squares (SOS) form

The electronic-structure Hamiltonian in second quantization is

$$
H=\sum_{pq}h_{pq}E_{pq}+\frac{1}{2}\sum_{pqrs}g_{pqrs}E_{pq}E_{rs}+E_{\mathrm{nuc}},
\qquad
E_{pq}=\sum_{\sigma}a_{p\sigma}^{\dagger}a_{q\sigma},
$$

with $N$ spatial orbitals. As written this is not a sum of squares, and the two-body tensor
$g_{pqrs}$ has $O(N^4)$ entries. Both problems are solved by factorizing $g$.

### Double-factorized tensor hypercontraction (DFTHC)

DFTHC replaces the two-body tensor with two smaller tensors  $u\in\mathbb{R}^{R\times B\times N}$ and
$w\in\mathbb{R}^{R\times(B+1)\times C}$

$$
H_{\mathrm{DFTHC}}=\sum_{pq}h_{pq}^{(1)\prime}E_{pq}
+\frac{1}{2}\sum_{r,c}\left(W^{(rc)}\mathbb{I}
+\sum_{b}w_{b}^{(rc)}\sum_{pq}u_{b,p}^{(r)}u_{b,q}^{(r)}E_{pq}\right)^{2}+\text{constant}.
$$

The size of the representation is controlled by three integers at
fixed $N$: the number of ranks $R$, the number of basis vectors $B$ inside each rank, and the
number of copies $C$ of each rank. 
The one-body matrix $h^{(1)\prime}$ is shifted by one-body contributions arising from the factorization of the two-body tensor.


Finding a compact, accurate DFTHC representation is a hard numerical optimization. 
Here we load a pre-optimized factorization of $\mathrm{H}_2$ with $N=2$, $R=2$, $B=2$, $C=1$.

### Double factorization (DF) as a special case of DFTHC

DFTHC is not a separate, unprecedented format. It contains conventional double factorization as a 
special case, so both representations can use the same downstream SOSSA code.

Conventional double factorization is DFTHC with the inner structure fixed.
It uses a Cholesky decomposition of the two-electron tensor and then diagonalizes each Cholesky matrix,

$$
g_{pq,p'q'}=\sum_{r=1}^{R_{\mathrm{DF}}}L_{pq}^{(r)}L_{p'q'}^{(r)},
\qquad
L_{pq}^{(r)}=\sum_{b=1}^{N}\epsilon_{b}^{(r)}u_{b,p}^{(r)}u_{b,q}^{(r)},
$$

with eigenvalues $\epsilon^{(r)}$ and orthonormal eigenvectors $u^{(r)}$. Matching
the two expressions term by term, ordinary double factorization is DFTHC restricted to
$C=1, B=N$. 
Tensor hypercontraction sits at the opposite corner, $(R,B,C)=(1,R_{\mathrm{THC}},R_{\mathrm{THC}})$, and
DFTHC interpolates between the two by varying $(R,B,C)$ freely.
The rank $R_{\mathrm{DF}}$ is not a free knob here: it is whatever the Cholesky truncation leaves
behind, set by `cholesky_tolerance`; the eigendecomposition that follows is exact.

The cell below runs that pipeline on $\mathrm{H}_2$ in the STO-3G basis. 
It is here to show that double factorization lands in the
same `DFTHCHamiltonianContainer` that everything downstream consumes.

### Synthetic factorizations for approximate resource estimation

The two routes above produce a factorization of a specific molecule. For resource estimation we
sometimes want the opposite: the cost of a system whose factorization parameters are published, without
redoing the factorization.

This works for resource counting because the compiled circuit mostly depends on the tensor *shapes* $(N,R,B,C)$ and the arithmetic
precisions, not so much on the tensor values. So sampling random tensors of the right shape can roughly reproduce the gate
counts.

The cell below does this for the 76-orbital $\mathrm{FeMoco}$ active space by drawing a symmetric one-body matrix,
unit-norm basis vectors $u$ of shape $R\times B\times N$, two-body weights $w$ of shape
$R\times B\times C$, and identity weights $w_B$ of shape $R\times C$ from a fixed seed. The query
count used later comes from the published $\lambda_{\mathrm{eff}}$.

In [2]:
import numpy as np
from qdk_chemistry.data import DFTHCHamiltonianContainer, ModelOrbitals
from qdk_chemistry.data import Hamiltonian


# Low et al. (2025), Table V, [113e, 76o] column (Li et al. active space);
# bit precisions from Table X (p. 32) and the p. 38 table, (·, 57, 19) column.
FEMoco_76_PARAMETERS = {
    "electrons": 113,
    "N": 76,
    "R": 15,
    "B": 57,
    "C": 19,
    "lambda_eff": 43.6538,
    "b_coeff": 9,
    "b_rot": 15,
}

n, r, b, c = (FEMoco_76_PARAMETERS[key] for key in ("N", "R", "B", "C"))
HAMILTONIAN_SEED = 42
rng = np.random.default_rng(HAMILTONIAN_SEED)
one_body = rng.standard_normal((n, n))
basis_vectors = rng.standard_normal((r, b, n))
basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)

synthetic_femoco_hamiltonian = Hamiltonian(
    DFTHCHamiltonianContainer(
        one_body_integrals=(one_body + one_body.T) / 2,
        u_matrices=basis_vectors.ravel(),
        w_matrices=0.1 * rng.standard_normal((r, b, c)).ravel(),
        wb_matrix=0.1 * rng.standard_normal((r, c)),
        orbitals=ModelOrbitals(n),
        core_energy=0.0,
        inactive_fock_matrix=np.zeros((n, n)),
    )
)
print(synthetic_femoco_hamiltonian.get_summary())

Hamiltonian Summary:
  Type: Hermitian
  Restrictedness: Restricted
  Active Orbitals: 76
  Total Orbitals: 76
  Core Energy: 0.000000
  Integral Statistics:
    One-body Integrals (alpha): 5776 (larger than 0.000001: 5776)
    Two-body Integrals (aaaa): 33362176 (larger than 0.000001: 33348476)



## From the factorized Hamiltonian to a qubit operator

Every factorized Hamiltonian above — loaded, double-factorized, or synthetic — now enters the same
pipeline. The first step is to read off the generators $O_\alpha$ that make the sum of squares
and to stack them into the rectangular square root that actually gets block-encoded,

$$
H=\sum_{\alpha}O_{\alpha}^{\dagger}O_{\alpha}+E_{\mathrm{SOS}}, \qquad H_{\mathrm{sqrt}}=\sum_{\alpha}|\alpha\rangle\otimes O_{\alpha}.
$$

DFTHC already provides the two-body part as squares. The mapper expresses their generators in forms
that can be encoded efficiently with rotated Pauli operators.

**Two-body: already a square.** Each rank-copy pair $(r,c)$ contributes one spin-free operator
counting electrons in the rotated orbitals $\boldsymbol u_b^{(r)}$:

$$
O_{\mathrm{SF},rc}
=\frac{1}{\sqrt2}\left(W^{(rc)}\mathbb{I}
+\sum_{b=0}^{B-1}w_{b}^{(rc)}\sum_{\sigma}n_{\boldsymbol u_{b}^{(r)}\sigma}\right).
$$

Here, $W^{(rc)}$ is the offset in the number-operator expression. The DFTHC container instead stores
$w_B^{(rc)}=W^{(rc)}+\sum_{b=0}^{B-1}w_b^{(rc)}$ as the corresponding identity coefficient.

**One-body: diagonalize, then split by sign.** Diagonalizing $h^{(1)\prime}$ gives positive
eigenpairs $(w_+^{(j)},\boldsymbol v_+^{(j)})$ and negative eigenpairs
$(-w_-^{(j)},\boldsymbol v_-^{(j)})$, where $w_+^{(j)},w_-^{(j)}\ge0$.

A positive eigenvalue produces a particle generator directly. A negative eigenvalue is rewritten
using $a a^{\dagger}=\mathbb{I}-a^{\dagger}a$, which produces a hole generator and a constant shift.
The corresponding generators are

$$
O_{\mathrm{D}_1,j\sigma}=\sqrt{w_{+}^{(j)}}\;a_{\boldsymbol v_{+}^{(j)}\sigma}
=\frac{\sqrt{w_+^{(j)}}}{2}
\left(\gamma_{\boldsymbol v_+^{(j)}\sigma0}+i\gamma_{\boldsymbol v_+^{(j)}\sigma1}\right),
\qquad
O_{\mathrm{Q}_1,j\sigma}=\sqrt{w_{-}^{(j)}}\;a_{\boldsymbol v_{-}^{(j)}\sigma}^{\dagger}
=\frac{\sqrt{w_-^{(j)}}}{2}
\left(\gamma_{\boldsymbol v_-^{(j)}\sigma0}-i\gamma_{\boldsymbol v_-^{(j)}\sigma1}\right).
$$

The index $j$ labels eigenvectors of $h^{(1)\prime}$ and has nothing to do with the DFTHC rank $r$.

**Constants: collect into $E_{\mathrm{SOS}}$.** The hole rewrite and the two-body offsets each leave
a constant behind. Gathered with the core energy they give

$$
E_{\mathrm{SOS}}=E_{\mathrm{core}}-2\sum_{j}w_{-}^{(j)}-\frac{1}{2}\sum_{r,c}\big|W^{(rc)}\big|^{2},
$$

the factor $2$ being the spin sum. $E_{\mathrm{SOS}}$ is added
back when a measured phase is converted into an energy.

**Map the generators to qubits.** After rotating the selected orbital onto the working mode, the
Jordan-Wigner mapping represents $a$ and $a^\dagger$ using $(X+iY)/2$ and $(X-iY)/2$, respectively,
while the number operator becomes $n=(\mathbb{I}-Z)/2$. Therefore, the mapper stores the one-body
generators as rotated $X\pm iY$ terms and the spin-free generators as rotated $Z$ terms plus an
identity term.

The `sum_of_squares` qubit mapper performs these steps and stores the rotation angles, coefficients,
and Pauli labels needed by the SOSSA circuit builder.

## SOSSA block encoding and quantum walk

The square root operator $\sum_{\alpha}|\alpha\rangle\otimes O_{\alpha}$ is block encoded in two levels of linear combination of unitaries (LCU):
an outer choice of *which* operator, and an inner choice of *which* term within that operator determined by the structure of $O_\alpha$.

**1. Outer PREPARE** An outer index $x_o\in[0,N+RC)$ enumerates the operators, the first
$N$ being the one-body $D_1/Q_1$ modes and the remaining $RC$ the spin-free $(r,c)$ generators

$$
\mathrm{PREP}_{\mathrm{out}}|0\rangle=\sum_{x_{o}}\sqrt{\frac{\lambda_{x_{o}}^{2}}{2\Lambda}}\,|x_{o}\rangle,
\qquad \textrm{with} \quad
\lambda_{D_{1}/Q_{1},j}=\sqrt{2w_{\pm}^{(j)}},
\quad
\lambda_{\mathrm{SF},rc}=\frac{\big|w_{B}^{(rc)}\big|+\sum_{b}\big|w_{b}^{(rc)}\big|}{\sqrt2},
\quad
\Lambda=\frac{1}{2}\sum_{x_{o}}\lambda_{x_{o}}^{2}.
$$

**2. Inner PREPARE**, controlled on $x_o$, an inner index $b\in[0,B]$ enumerates the terms within the chosen operator:

$$
\mathrm{PREP}_{\mathrm{in}}|x_{o}\rangle|0\rangle=|x_{o}\rangle\sum_{b=0}^{B}\sqrt{\frac{\big|w_{b}^{(x_{o})}\big|}{\sum_{b'}\big|w_{b'}^{(x_{o})}\big|}}\,|b\rangle\otimes|+\rangle.
$$

For a spin-free row, $b<B$ are the orbital terms and $b=B$ addresses the identity weight. A
$D_1/Q_1$ row has a single term at $b=0$. The sign of each $w_b$ is applied inline. 

Since the two-body term is spin-free, the $|+\rangle$ selects the spin for spin-free
rows, while a second $|+\rangle$ prepared alongside the outer
index selects the spin for $D_1/Q_1$.

**3. SELECT** applies the chosen term. It swaps the selected spin into the working register
($S_\sigma$), rotates the orbital $\boldsymbol u_{x_ob}$ onto the first qubit ($G_{x_ob}$), applies a
single Pauli there, and undoes both:

$$
\mathrm{SELECT}=\sum_{x_{o},b,\sigma}|x_{o},b,\sigma\rangle\langle x_{o},b,\sigma|\otimes(\pm1)\,S_{\sigma}^{\dagger}G_{x_{o}b}^{\dagger}P_{x_{o}b}G_{x_{o}b}S_{\sigma},
\qquad
P_{x_{o}b}\in\{X,\pm iY\}_{D_{1}/Q_{1}},\ \{Z\}_{\mathrm{SF},b<B},\ \{I\}_{\mathrm{SF},b=B}.
$$

The $\pm1$ is the coefficient sign, and $+iY$ versus $-iY$ distinguishes $a=(X+iY)/2$ from
$a^{\dagger}=(X-iY)/2$.

**Assembling the walk.** Define

$$
U=\mathrm{PREP}_{\mathrm{in}}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREP}_{\mathrm{in}}\,\mathrm{PREP}_{\mathrm{out}},
$$

where operators act from right to left: first prepare the outer index, then prepare the inner index,
apply SELECT, and unprepare the inner index.
Reflecting about the inner ancillas gives a self-inverse block encoding of the shifted, rescaled
Hamiltonian,

$$
\mathcal{B}=U^{\dagger}\mathrm{Ref}_{\mathcal{B}}U=\operatorname{BE}\!\left[\frac{H-E_{\mathrm{SOS}}}{\Lambda}-1\right],
\qquad
\mathcal{B}^{2}=\mathbb{I},
$$

and the walk is $W=\mathrm{Ref}_{a\mathcal{B}}\,\mathcal{B}$, whose eigenphases $\theta$ satisfy

$$
\cos\theta=\frac{E-E_{\mathrm{SOS}}}{\Lambda}-1 .
$$

A single reflection maps the right singular
vectors of $H_{\mathrm{sqrt}}$ to the left ones, so it has no invariant subspace to iterate on. The
operator that is actually phase-estimated is the *two-step* product above, which means each walk
step invokes the block encoding twice, once as $U$ and once as $U^{\dagger}$.

In [3]:
from qdk_chemistry.data import MajoranaMapping
from qdk_chemistry.algorithms import create

def map_to_sum_of_squares(factorized_hamiltonian):
    num_orbitals = factorized_hamiltonian.get_container().get_num_orbitals()

    return create("qubit_mapper", "sum_of_squares").run(
        factorized_hamiltonian,
        MajoranaMapping.jordan_wigner(2 * num_orbitals)
    )

## Mapping to a circuit with different primitives

The walk is still an abstract sequence of PREPARE, SELECT, and reflection. The circuit
mapper can implement those operations with primitives that trade qubit count, runtime, and fidelity,
depending on whether the
goal is small-system validation or fault-tolerant resource estimation.

- Validation primitives, designed for state-vector simulation of small examples:
  - dense state preparation loads the outer distribution:
    `outer_prepare_algorithm=AlgorithmRef("state_prep", "dense_pure_state")`;
  - direct multiplexed rotations load the conditional inner distributions:
    `inner_prepare_algorithm="direct"`; and
  - direct SELECT applies each Givens angle as an exact rotation:
    `select_algorithm="direct"`.

This choice keeps the ancillary-qubit count small, but its gate count does not scale well. It is a
correctness check, not a fault-tolerant cost model.

- Scalable resource-estimation primitives:
  - alias sampling prepares the outer distribution:
    `outer_prepare_algorithm=AlgorithmRef("state_prep", "alias_sampling")`;
  - controlled alias sampling prepares the inner distributions using $b_{\mathrm{coeff}}$-bit
    coefficient tables: `inner_prepare_algorithm="controlled_alias_sampling"`; and
  - QROM loads each orbital rotation as $(N-1)b_{\mathrm{rot}}$ angle bits plus one sign bit, and a
    shared phase-gradient register applies it: `select_algorithm="qrom_phase_gradient"`.

These finite-precision lookup and arithmetic operations provide a realistic fault-tolerant circuit,
but they require many more logical qubits than what the simulator can handle.

Use the validation primitives only for simulation, and use the scalable primitives when estimating
fault-tolerant resources.

In [4]:
from qdk_chemistry.data import AlgorithmRef

validation_mapper = AlgorithmRef(
    "circuit_mapper",
    "sossa",
    outer_prepare_algorithm=AlgorithmRef("state_prep", "dense_pure_state"),
    inner_prepare_algorithm="direct",
    select_algorithm="direct",
)

## Preparing a trial state

Phase estimation returns the ground-state energy only in proportion to how much the input state
overlaps the ground state; with overlap $\gamma$ the correct phase is measured with probability
$|\gamma|^2$. 

For $\mathrm{H}_2$ in a minimal basis the Hartree-Fock determinant is already an excellent
approximation, so we prepare the canonical one by sparse isometry. 
Strongly correlated systems need
a multi-configuration trial state instead — the [stretched $\mathrm{N}_2$ notebook](qpe_stretched_n2.ipynb) demonstrates that workflow.

In [5]:
from qdk_chemistry.data import Configuration, StateVectorContainer, Wavefunction


def prepare_hartree_fock(factorized_hamiltonian, num_alpha, num_beta):
    """Prepare the canonical Hartree-Fock determinant by sparse isometry."""
    container = factorized_hamiltonian.get_container()
    configuration = Configuration.canonical_hf_configuration(
        num_alpha, num_beta, container.get_num_orbitals()
    )
    reference = Wavefunction(
        StateVectorContainer(configuration, container.get_orbitals())
    )
    return create("state_prep", "sparse_isometry").run(reference)

## Resource estimation for $\mathrm{FeMoco}$ 76 orb (Li et al. active space, 113 electrons)

We now estimate the fault-tolerant resources for the published $\mathrm{FeMoco}$
model. We use its reported DFTHC dimensions, effective normalization, and precision
settings.

The tensor values are synthetic. Their dimensions reproduce the circuit structure needed for
resource estimation, but their spectrum and energies do not represent $\mathrm{FeMoco}$.

Assuming the initial state preparation cost is negligible compared with the phase estimation cost, we
use a Hartree-Fock determinant as a placeholder trial state. And we use the scalable resource-estimation 
primitives described above.

In [6]:
import math

sossa_unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "sossa")

TARGET_PRECISION = 1e-3

femoco76_operator = map_to_sum_of_squares(synthetic_femoco_hamiltonian)
femoco76_state_preparation = prepare_hartree_fock(
    synthetic_femoco_hamiltonian,
    (FEMoco_76_PARAMETERS["electrons"] + 1) // 2,
    FEMoco_76_PARAMETERS["electrons"] // 2,
)
femoco76_num_queries = math.ceil(
    math.pi * FEMoco_76_PARAMETERS["lambda_eff"] / (2 * TARGET_PRECISION)
)
femoco76_circuit = create(
    "qpe_circuit_builder",
    "qdk_unary",
    num_queries=femoco76_num_queries,
    circuit_mapper=AlgorithmRef(
        "circuit_mapper",
        "sossa",
        outer_prepare_algorithm=AlgorithmRef("state_prep", "alias_sampling"),
        inner_prepare_algorithm="controlled_alias_sampling",
        select_algorithm="qrom_phase_gradient",
        rotation_bit_precision=FEMoco_76_PARAMETERS["b_rot"],
        coefficient_bit_precision=FEMoco_76_PARAMETERS["b_coeff"],
    ),
    unitary_builder=sossa_unitary_builder,
).run(
    state_preparation=femoco76_state_preparation,
    qubit_hamiltonian=femoco76_operator,
)[0]

logical_counts = femoco76_circuit.estimate().logical_counts
toffoli_count = logical_counts["cczCount"] + logical_counts["ccixCount"]
print(f"Walk steps:     {femoco76_num_queries:,}")
print(f"Logical counts: {logical_counts}")

Walk steps:     68,572
Logical counts: {'numQubits': 1490, 'tCount': 132, 'rotationCount': 15217817, 'rotationDepth': 14599948, 'cczCount': 1092626301, 'ccixCount': 0, 'measurementCount': 1223735982}


### Physical resource estimation

Here, we use [QRE](https://github.com/microsoft/qdk/tree/main/samples/qre) to evaluate compatible
code, factory, and trace choices and return their
Pareto-optimal physical-qubit and runtime tradeoffs. The [stretched $\mathrm{N}_2$ notebook](qpe_stretched_n2.ipynb) introduces the same QRE workflow, with the following parameters:

- `femoco76_circuit.get_qre_application()` converts the compiled Q# circuit into the logical
  application trace that QRE analyzes.
- `Gate based(error_rate=1e-3)` selects a gate-based physical architecture and assigns a
  $10^{-3}$ error rate to its physical operations.
- `SurfaceCode.q(distance=27, code_cycle_override=1000)` selects the surface code error-correction scheme with code distance 27 and imposes a logical cycle time of 1000ns for logical computation.
- `RoundBasedFactory.q(...)` selects round-based magic-state factories.
- `PSSPC.q(num_ts_per_rotation=NUM_TS_PER_ROTATION) * LatticeSurgery.q()` forms the `trace_query`: `PSSPC` synthesizes arbitrary
  rotations, using 5 to 40 T states each (QRE's default upper limit of 20 cannot meet the error budget for this circuit's ~1.5e7 rotations), and `LatticeSurgery` maps the resulting logical operations to lattice-surgery
  instructions.
- `MAX_ERROR = 0.01` limits the total allowed failure probability of the computation to 1%.

In [7]:
# Shared physical-architecture assumptions for both estimates below.
# (Named constants so the machine-written results.json reads the same values the estimator used.)
ARCHITECTURE_PARAMS = dict(error_rate=1e-3, gate_time=100, measurement_time=500)  # ns
CODE_PARAMS = dict(distance=27, code_cycle_override=1000)                          # surface code, 1 µs cycle
ERROR_BUDGET = 0.01

# Search range for the number of T states PSSPC spends synthesizing each arbitrary rotation.
# QRE's default range is 5-20. This 76-orbital circuit has about 1.5e7 rotations, and at 20 T
# states each their synthesis error alone is about 0.038, above ERROR_BUDGET, so no estimate
# exists in the default range. Widening the range lets the estimator spend more T states per
# rotation only where the budget requires it. The 54-orbital record fits in the default range.
NUM_TS_PER_ROTATION = list(range(5, 41))


In [8]:
from qdk.qre import PSSPC, LatticeSurgery, estimate, property_name_to_key

from qdk.qre.models import (
    GateBased,
    SurfaceCode,
    RoundBasedFactory,
    GSJ24CCXFactory,
)

architecture = GateBased(**ARCHITECTURE_PARAMS)

surface_code = SurfaceCode.q(**CODE_PARAMS)

isa_query = (
    surface_code
    * RoundBasedFactory.q(
        use_cache=True,
        code_query=SurfaceCode.q(
            code_cycle_override=1000,
        ),
    )
    * GSJ24CCXFactory.q()
)

trace_query = (
    PSSPC.q(ccx_magic_states=True, num_ts_per_rotation=NUM_TS_PER_ROTATION)
    * LatticeSurgery.q()
)

results = estimate(
    femoco76_circuit.get_qre_application(),
    architecture,
    isa_query,
    trace_query,
    max_error=ERROR_BUDGET,
    name="SOSSA FeMoco 76",
)


results.add_factory_summary_column()
results.add_qubit_partition_column()
results.add_property_column(property_name_to_key("NUM_TS_PER_ROTATION"), "num_ts_per_rotation")

display(results.as_frame())

results_ccx_factories = results


,name,qubits,runtime,error,factories,physical_compute_qubits,physical_factory_qubits,physical_memory_qubits,num_ts_per_rotation
0,SOSSA FeMoco 76,4676575,1 days 12:23:41.054226,0.008027,"1×T, 7×CCX",4503587,172988,0,23
1,SOSSA FeMoco 76,4680095,1 days 12:17:06.855630,0.009377,"1×T, 7×CCX",4503587,176508,0,22


In [9]:
import inspect
from qdk.qre.models import GSJ24CCXFactory

print(inspect.getdoc(GSJ24CCXFactory))

Implements the 8|T⟩ → |CCX⟩ magic state factory described in Fig. 24 of
Gidney, Shutty, and Jones (2024). This design converts eight T magic
states into a single CCX (Toffoli) state using lattice surgery operations
on 12 logical qubits (including helper qubits) with a circuit depth of 6.

The output CCX error rate has two contributions:

- Distillation error: 28 · p_T², where p_T is the T state error rate
  (from pairs of T states failing simultaneously).
- Logical error: accumulated over 6 lattice surgery rounds on 12 qubits.

The factory production time includes an overhead factor of (1 + 8·p_T) to
account for the failure probability when consuming the T states.

Reference:
    - C. Gidney, C. Shutty, C. Jones, "Magic state cultivation: growing
      T states with 78% reduced overhead", arXiv:2409.17595 (2024).
      https://arxiv.org/abs/2409.17595
    - C. Gidney, A. G. Fowler, "Efficient magic state factories with a
      catalyzed |CCZ⟩ to 2|T⟩ transformation", Quantum 3, 135 (20

### Physical resource estimation

Here, we use [QRE](https://github.com/microsoft/qdk/tree/main/samples/qre) to evaluate compatible
code, factory, and trace choices and return their
Pareto-optimal physical-qubit and runtime tradeoffs. The [stretched $\mathrm{N}_2$ notebook](qpe_stretched_n2.ipynb) introduces the same QRE workflow, with the following parameters:

- `femoco76_circuit.get_qre_application()` converts the compiled Q# circuit into the logical
  application trace that QRE analyzes.
- `Gate based(error_rate=1e-3)` selects a gate-based physical architecture and assigns a
  $10^{-3}$ error rate to its physical operations.
- `SurfaceCode.q(distance=27, code_cycle_override=1000)` selects the surface code error-correction scheme with code distance 27 and imposes a logical cycle time of 1000ns for logical computation.
- `Litinski19Factory.q(...)` follows Litinski's 2019 work on magic state distillation.
- `PSSPC.q(num_ts_per_rotation=NUM_TS_PER_ROTATION) * LatticeSurgery.q()` forms the `trace_query`: `PSSPC` synthesizes arbitrary
  rotations, using 5 to 40 T states each (QRE's default upper limit of 20 cannot meet the error budget for this circuit's ~1.5e7 rotations), and `LatticeSurgery` maps the resulting logical operations to lattice-surgery
  instructions.
- `MAX_ERROR = 0.01` limits the total allowed failure probability of the computation to 1%.

In [10]:
from qdk.qre import PSSPC, LatticeSurgery, estimate, property_name_to_key

from qdk.qre.models import (
    GSJ24Factory,
    GateBased,
    Litinski19Factory,
    SurfaceCode,
    RoundBasedFactory,
    GSJ24CCXFactory,
)

architecture = GateBased(**ARCHITECTURE_PARAMS)

surface_code = SurfaceCode.q(**CODE_PARAMS)
isa_query = (
    surface_code
    * Litinski19Factory.q()
)

trace_query = (
    PSSPC.q(ccx_magic_states=False, num_ts_per_rotation=NUM_TS_PER_ROTATION)
    * LatticeSurgery.q()
)

results = estimate(
    femoco76_circuit.get_qre_application(),
    architecture,
    isa_query,
    trace_query,
    max_error=ERROR_BUDGET,
    name="SOSSA FeMoco 76",
)


results.add_factory_summary_column()
results.add_qubit_partition_column()
results.add_property_column(property_name_to_key("NUM_TS_PER_ROTATION"), "num_ts_per_rotation")

display(results.as_frame())

results_t_factories = results


,name,qubits,runtime,error,factories,physical_compute_qubits,physical_factory_qubits,physical_memory_qubits,num_ts_per_rotation
0,SOSSA FeMoco 76,4659987,1 days 12:17:06.855630,0.007442,4×T,4503587,156400,0,22


## Machine-readable results for the QAC claim record

Everything the public record attributes to this reproduction is written here, from the live objects above —
logical counts, both physical scenarios, the inputs they used, and the exact package versions.
The cell is tagged `results-export`; `qacclaims validate` checks that the committed `results.json`
is byte-identical to what this notebook produces in its pinned environment.


In [11]:
import json
import platform
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata

CLAIM_ID = "femoco-low-2025-li76"
REPRODUCTION_ID = "qdk-sossa-2026-11"


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "unknown"


def scenario(results_object, scenario_id, synthesis):
    row = results_object.as_frame().iloc[0]
    return {
        "scenario_id": scenario_id,
        "architecture": {
            "physical_error_rate": ARCHITECTURE_PARAMS["error_rate"],
            "gate_time_ns": ARCHITECTURE_PARAMS["gate_time"],
            "measurement_time_ns": ARCHITECTURE_PARAMS["measurement_time"],
        },
        "code": {
            "family": "surface",
            "distance": CODE_PARAMS["distance"],
            "distance_source": "imposed",
            "cycle_time_ns": CODE_PARAMS["code_cycle_override"],
        },
        "factories": str(row["factories"]),
        "synthesis": f"{synthesis}; the estimator chose {int(row['num_ts_per_rotation'])} T states per rotation",
        "physical_qubits": int(row["qubits"]),
        "compute_qubits": int(row["physical_compute_qubits"]),
        "factory_qubits": int(row["physical_factory_qubits"]),
        "memory_qubits": int(row["physical_memory_qubits"]),
        "runtime_seconds": float(row["runtime"].total_seconds()),
        "error_budget": ERROR_BUDGET,
        "achieved_error": float(row["error"]),
    }


results_record = {
    "schema_version": "1.0",
    "claim_id": CLAIM_ID,
    "reproduction_id": REPRODUCTION_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "producer": {"entrypoint": "notebook.ipynb", "cell_tag": "results-export"},
    "tooling": {
        "python": platform.python_version(),
        "packages": {name: package_version(name) for name in ("qdk-chemistry", "qdk", "numpy", "scipy")},
    },
    "inputs": {
        "hamiltonian": {
            "kind": "synthetic_dfthc",
            "orbitals": FEMoco_76_PARAMETERS["N"],
            "electrons": FEMoco_76_PARAMETERS["electrons"],
            "R": FEMoco_76_PARAMETERS["R"],
            "B": FEMoco_76_PARAMETERS["B"],
            "C": FEMoco_76_PARAMETERS["C"],
            "lambda_eff": FEMoco_76_PARAMETERS["lambda_eff"],
            "coefficient_bit_precision": FEMoco_76_PARAMETERS["b_coeff"],
            "rotation_bit_precision": FEMoco_76_PARAMETERS["b_rot"],
            "seed": HAMILTONIAN_SEED,
        },
        "target_precision_hartree": TARGET_PRECISION,
        "trial_state": "hartree_fock_placeholder",
    },
    "logical": {
        "walk_steps": int(femoco76_num_queries),
        "num_qubits": int(logical_counts["numQubits"]),
        "t_count": int(logical_counts["tCount"]),
        "rotation_count": int(logical_counts["rotationCount"]),
        "rotation_depth": int(logical_counts["rotationDepth"]),
        "ccz_count": int(logical_counts["cczCount"]),
        "ccix_count": int(logical_counts["ccixCount"]),
        "measurement_count": int(logical_counts["measurementCount"]),
        "toffoli_count": int(logical_counts["cczCount"]) + int(logical_counts["ccixCount"]),
    },
    "physical": [
        scenario(results_ccx_factories, "gsj24-ccx-factories", "PSSPC (5-40 T states per rotation) with CCX magic states; round-based T factory feeding GSJ24 8T->CCX factory; lattice surgery"),
        scenario(results_t_factories, "litinski19-t-factories", "PSSPC (5-40 T states per rotation) with T magic states; Litinski 2019 T factories; lattice surgery"),
    ],
}

with open("results.json", "w", encoding="utf-8") as handle:
    json.dump(results_record, handle, indent=2)
    handle.write("\n")

print(json.dumps(results_record["logical"], indent=2))
print({s["scenario_id"]: (s["physical_qubits"], round(s["runtime_seconds"])) for s in results_record["physical"]})


{
  "walk_steps": 68572,
  "num_qubits": 1490,
  "t_count": 132,
  "rotation_count": 15217817,
  "rotation_depth": 14599948,
  "ccz_count": 1092626301,
  "ccix_count": 0,
  "measurement_count": 1223735982,
  "toffoli_count": 1092626301
}
{'gsj24-ccx-factories': (4676575, 131021), 'litinski19-t-factories': (4659987, 130627)}
